# 🧬 SiamProm: Cyanobacterial Promoter Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Passion4ever/SiamProm/blob/main/notebooks/SiamProm_Colab.ipynb)

Predict whether 81 bp DNA sequences are cyanobacterial promoters.
Paper: *Recognition of Cyanobacteria Promoters via Siamese Network-based Contrastive Learning under Novel Non-promoter Generation* · [Code](https://github.com/Passion4ever/SiamProm)

**How to use**
1. Pick a GPU: *Runtime → Change runtime type → T4 GPU* (a few sequences also run fine on CPU).
2. Run the cells from top to bottom with the ▶ button.
3. In Step 3, choose Example, Paste, or Upload FASTA.

**Input:** 81 bp DNA (A/T/C/G, any case), as FASTA or one sequence per line. Sequences of any other length, or with other characters, are not predicted and are listed with the reason.


In [ ]:
#@title 🔧 Step 1 · Setup
import os, subprocess, sys

REPO_URL = "https://github.com/Passion4ever/SiamProm.git"
REPO_DIR = "/content/SiamProm"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
from srcs import colab as sc

device = sc.pick_device()
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow)")


In [ ]:
#@title 🎛️ Step 2 · Choose model
if "sc" not in globals():
    raise RuntimeError("Run Step 1 first.")

import ipywidgets as widgets
from IPython.display import display

registry = sc.load_registry()
# Remembered choices. Widgets are created fresh on every run (the browser may have
# forgotten old ones after a refresh); these values make re-running keep your selection.
prefs = globals().setdefault("prefs", {})

options = [(f"{e['name']} - {e['description']}", e["name"]) for e in registry]
if len(registry) > 1:
    options.append(("All models (compare)", "__all__"))
names = [value for _, value in options]

model_dd = widgets.Dropdown(
    options=options, value=prefs["model"] if prefs.get("model") in names else names[0],
    description="Model:", layout=widgets.Layout(width="80%"))
# Used for a single model; "All models" uses each model's own default threshold.
threshold_sl = widgets.FloatSlider(
    value=prefs.get("threshold", float(registry[0]["threshold"])), min=0.01, max=0.99,
    step=0.01, description="Threshold:")
batch_in = widgets.BoundedIntText(value=prefs.get("batch", 1024), min=1, max=65536,
                                  description="Batch size:")


def _sync_threshold(change):
    for e in registry:
        if e["name"] == change["new"]:
            threshold_sl.value = float(e["threshold"])


model_dd.observe(_sync_threshold, names="value")
model_dd.observe(lambda c: prefs.update(model=c["new"]), names="value")
threshold_sl.observe(lambda c: prefs.update(threshold=c["new"]), names="value")
batch_in.observe(lambda c: prefs.update(batch=c["new"]), names="value")

display(model_dd, threshold_sl, batch_in)


In [ ]:
#@title 🧪 Step 3 · Input sequences
if "model_dd" not in globals():
    raise RuntimeError("Run Step 2 first.")

HINTS = {
    "example": "<b>Example</b> &middot; 10 known promoters + 10 randomly generated non-promoters. "
               "A quick demo of the workflow, not a benchmark.",
    "paste": "<b>Paste</b> &middot; FASTA, or one 81 bp sequence per line.",
    "upload": "<b>Upload FASTA</b> &middot; a file picker will appear when you run Step 4.",
}

input_mode = widgets.ToggleButtons(
    options=[("Example", "example"), ("Paste", "paste"), ("Upload FASTA", "upload")],
    value=prefs.get("source", "example"), description="Source:",
    style={"button_width": "130px"})
paste_box = widgets.Textarea(
    value=prefs.get("paste", ""), placeholder=">seq1\nATCG... (81 bp)\nor one sequence per line",
    layout=widgets.Layout(width="90%", height="160px"))
hint = widgets.HTML()


def _refresh(_=None):
    paste_box.layout.display = "block" if input_mode.value == "paste" else "none"
    hint.value = f'<div style="opacity:0.8;margin:4px 0">{HINTS[input_mode.value]}</div>'


input_mode.observe(_refresh, names="value")
input_mode.observe(lambda c: prefs.update(source=c["new"]), names="value")
paste_box.observe(lambda c: prefs.update(paste=c["new"]), names="value")

_refresh()
display(input_mode, hint, paste_box)


In [ ]:
#@title 🚀 Step 4 · Run prediction
if "input_mode" not in globals():
    raise RuntimeError("Run Step 3 first.")

import os

globals().pop("out", None)  # never show old results if this run fails
for f in ("siamprom_results.csv", "siamprom_skipped.csv"):  # nor offer old files for download
    if os.path.exists(f):
        os.remove(f)

from tqdm.auto import tqdm

mode = input_mode.value
if mode == "example":
    names, seqs = sc.load_example()
elif mode == "paste":
    names, seqs = sc.parse_sequences(paste_box.value)
else:
    from google.colab import files
    uploaded = files.upload()
    names, seqs = [], []
    for _, data in uploaded.items():
        n, s = sc.parse_sequences(data.decode("utf-8-sig", errors="replace"))
        names += n
        seqs += s
if not seqs:
    raise ValueError("No sequences were read. Check your input in Step 3.")
print(f"Source: {dict(example='Example', paste='Paste', upload='Upload FASTA')[mode]} \u00b7 {len(seqs)} sequences read")

selected = [e for e in registry if model_dd.value in ("__all__", e["name"])]
models = {
    e["name"]: sc.load_model_safe(sc.resolve_checkpoint(e), device,
                                  allow_pickle=e.get("allow_pickle", False))
    for e in selected
}
thresholds = {e["name"]: (threshold_sl.value if len(selected) == 1 else float(e["threshold"]))
              for e in selected}

bar = tqdm(unit="seq")


def progress(done, total):
    bar.total = total
    bar.n = done
    bar.refresh()


out = sc.run_prediction(names, seqs, models, device, batch_size=batch_in.value,
                        thresholds=thresholds, progress=progress)
bar.close()


In [ ]:
#@title 📊 Step 5 · Results
if "out" not in globals():
    raise RuntimeError("Run Step 4 first.")

from IPython.display import HTML, display
import os

for f in ("siamprom_results.csv", "siamprom_skipped.csv"):  # drop files from an earlier run
    if os.path.exists(f):
        os.remove(f)

res, skipped, stats = out["results"], out["skipped"], out["stats"]
display(HTML(sc.summary_html(stats)))

if stats["n_skipped"]:
    print("Not predicted (sequences must be exactly 81 bp and contain only A/T/C/G):")
    display(skipped.head(50))
    skipped.to_csv("siamprom_skipped.csv", index=False)
    print("Saved siamprom_skipped.csv")

if res.empty:
    print("Nothing to predict.")
else:
    display(sc.style_results(res.head(50)))
    if len(res) > 50:
        print(f"Showing the first 50 rows; all {len(res)} rows are in the downloaded CSV.")

    res.to_csv("siamprom_results.csv", index=False)
    print("Saved siamprom_results.csv (Files panel on the left). To download it, run the next cell.")


In [ ]:
#@title 💾 Optional · Download results
import os
from google.colab import files

saved = [f for f in ("siamprom_results.csv", "siamprom_skipped.csv") if os.path.exists(f)]
if not saved:
    print("Nothing to download yet. Run Step 4 (prediction) and Step 5 (results) first.")
for f in saved:
    files.download(f)
    print(f"\u2705 Download started: {f}")


In [ ]:
#@title ✅ Optional · Self-check
if "registry" not in globals():
    raise RuntimeError("Run Steps 1 and 2 first.")

# Confirms this notebook gives the same probabilities as the official predict.py.
sc.self_check(registry[0], device)
